# NB02 — BTCS v3: Budgeted Temporal Case Segmentation
**Andre da Costa Silva | ITA | 2026**

Algoritmo proposto para ICDM 2026 — abordagem hierarquica:
1. **WCC**: componentes conexos no subgrafo top-k (cobertura alta, = B0)
2. **Leiden**: refinar apenas componentes grandes (|induzidas| > B) usando Lk temporal
3. **Budget B**: limitar arestas induzidas por caso a no maximo B

Vantagens vs Leiden puro (v2):
- Cobertura preservada para componentes pequenos (= B0)
- Leiden so atua onde necessario (componentes que excedem B)
- OCR(B) = 0 por construcao

Executar: celulas 1->2->3->4->5->6->7->8->9 (ou Runtime->Run all)

In [ ]:
# CELULA 1 - Setup: imports, paths, verificacao
import os, sys, subprocess, time, math, contextlib
from pathlib import Path
from collections import defaultdict

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

_pip('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'psutil', 'tqdm',
     'python-igraph', 'leidenalg')
try:
    import torch
except ImportError:
    _pip('torch', '--index-url', 'https://download.pytorch.org/whl/cu121')
    import torch

import numpy as np
import pandas as pd
import psutil
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__} | igraph {ig.__version__} | leidenalg {leidenalg.version}')
print(f'device: {DEVICE}')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(f'[WARN] Drive: {e}')

BASE = Path('/content/drive/MyDrive') if IN_COLAB else Path('.').resolve()

AML100K_BASE  = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML100k'
AML100K_ARTIF = AML100K_BASE / 'artifacts'
AML100K_PROBS = AML100K_BASE / 'results/probs_v4'
AML100K_MODEL = 'SAGE'
AML100K_SEED  = 42

AML1M_BASE    = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M'
AML1M_ARTIF   = AML1M_BASE / 'artifacts'
AML1M_PROBS   = AML1M_BASE / 'results_aml1m_graphsage_only/probs_v4'
AML1M_MODEL   = 'GraphSAGE'
AML1M_SEED    = 44

BTCS_OUT = BASE / 'GrafosGNN/results/nb02_btcs'
BTCS_OUT.mkdir(parents=True, exist_ok=True)
NB01_OUT = BASE / 'GrafosGNN/results/nb01_baselines'

print('\n=== Verificacao ===')
for label, path in [
    ('AML100k cache', AML100K_ARTIF / 'edge_data_v4_clean.pt'),
    ('AML100k probs', AML100K_PROBS / f'{AML100K_MODEL}_seed{AML100K_SEED}_test.npz'),
    ('AML1M   cache', AML1M_ARTIF   / 'edge_data_v4_clean.pt'),
    ('AML1M   probs', AML1M_PROBS   / f'{AML1M_MODEL}_seed{AML1M_SEED}_test.npz'),
    ('nb01 results', NB01_OUT / 'b0_b1_b2_b3_results.csv'),
]:
    print(f"  {'ok' if path.exists() else 'MISS'}  {label}")

In [ ]:
# CELULA 2 - Funcoes base (reutilizadas do nb01)

@contextlib.contextmanager
def measure_resources():
    proc = psutil.Process(os.getpid())
    m0 = proc.memory_info().rss / 1024**2
    t0 = time.perf_counter()
    r  = {}
    yield r
    r['time_s'] = time.perf_counter() - t0
    r['ram_mb']  = proc.memory_info().rss / 1024**2 - m0

def evaluate_cases(cases, gt, ranked, k):
    y_full = np.asarray(gt['y_full'], dtype=int)
    y_test = np.asarray(ranked['y'],  dtype=int)
    pos_te = int(y_test.sum())
    K      = max(1, int(math.ceil(k * len(y_test))))
    nan_row = {m: np.nan for m in [
        'n_cases','coverage','purity_induced','cr_at_k','recall_in_cases',
        'edges_per_case_median','edges_per_case_p95','edges_per_case_max',
        'e_ind_total','e_ind_over_ek','ocr_b100','ocr_b500','ocr_b1000']}
    if not cases:
        return nan_row
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    non_empty = [c['induced_edges'] for c in cases if c['induced_edges']]
    if non_empty:
        all_ind = np.unique(np.concatenate(
            [np.asarray(x, dtype=np.int64) for x in non_empty]))
    else:
        all_ind = np.array([], dtype=np.int64)
    pos_ind  = int(y_full[all_ind].sum()) if len(all_ind) > 0 else 0
    coverage = float(pos_ind / max(pos_te, 1))
    purity   = float(pos_ind / max(int(ind_sizes.sum()), 1))
    pos_sel  = sum(int(y_test[c['seed_edges']].sum()) for c in cases if c['seed_edges'])
    recall   = float(pos_sel / max(pos_te, 1))
    cr_at_k  = float(recall / max(coverage, 1e-12)) if coverage > 0 else np.nan
    return {'n_cases':len(cases), 'coverage':coverage, 'purity_induced':purity,
            'cr_at_k':cr_at_k, 'recall_in_cases':recall,
            'edges_per_case_median':float(np.median(ind_sizes)),
            'edges_per_case_p95':float(np.percentile(ind_sizes, 95)),
            'edges_per_case_max':float(ind_sizes.max()),
            'e_ind_total':int(ind_sizes.sum()),
            'e_ind_over_ek':float(ind_sizes.sum() / max(K, 1)),
            'ocr_b100':float((ind_sizes>100).mean()),
            'ocr_b500':float((ind_sizes>500).mean()),
            'ocr_b1000':float((ind_sizes>1000).mean())}

def load_dataset_artifacts(artif_dir, probs_dir, model, seed, dataset_name=''):
    artif_dir = Path(artif_dir); probs_dir = Path(probs_dir)
    npz  = np.load(probs_dir / f'{model}_seed{seed}_test.npz')
    p_te = np.asarray(npz['p'], dtype=float)
    y_te = np.asarray(npz['y'], dtype=int)
    print(f'[{dataset_name}] {len(p_te):,} arestas teste, {y_te.sum():,} pos ({100*y_te.mean():.2f}%)')
    cache  = torch.load(artif_dir/'edge_data_v4_clean.pt', map_location='cpu', weights_only=False)
    ei_all = cache['ei_all_cpu']
    te_idx = cache['te_idx']
    if isinstance(te_idx, torch.Tensor): te_idx = te_idx.numpy()
    if isinstance(ei_all, torch.Tensor): ei_all = ei_all.numpy()
    src_te = ei_all[0, te_idx]; dst_te = ei_all[1, te_idx]
    assert len(p_te)==len(src_te), f'Mismatch: {len(p_te)} vs {len(src_te)}'
    print(f'[{dataset_name}] Cache: {ei_all.shape[1]:,} totais, {len(te_idx):,} teste')
    return ({'scores':p_te,'y':y_te,'src':src_te,'dst':dst_te},
            {'src':src_te,'dst':dst_te},
            {'y_full':y_te}, te_idx, ei_all)

def load_timestamps_from_csv(data_dir, te_idx, ei_all):
    data_dir = Path(data_dir)
    candidates = ['transactions.csv','transaction.csv',
                  'hi-large_trans.csv','hi-medium_trans.csv','hi-small_trans.csv',
                  'li-large_trans.csv','li-medium_trans.csv','li-small_trans.csv']
    csv_path = next((list(data_dir.rglob(c))[0] for c in candidates
                     if list(data_dir.rglob(c))), None)
    if csv_path is None:
        raise FileNotFoundError(f'CSV nao encontrado em {data_dir}')
    print(f'  CSV: {csv_path.name}')
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    time_col = next((c for c in ['timestamp','time','date','datetime','step'] if c in df.columns), None)
    src_col  = next((c for c in ['sender_account_id','src','source','from','sender','src_id'] if c in df.columns), None)
    dst_col  = next((c for c in ['receiver_account_id','dst','dest','to','receiver','dst_id'] if c in df.columns), None)
    if time_col is None:
        raise KeyError(f'Coluna tempo nao encontrada. Cols: {list(df.columns)}')
    print(f'  Colunas: time={time_col!r}  src={src_col!r}  dst={dst_col!r}')
    if pd.api.types.is_numeric_dtype(df[time_col]):
        ts_raw = pd.to_numeric(df[time_col], errors='coerce').fillna(0).astype(np.int64).values
    else:
        ts_raw = (pd.to_datetime(df[time_col], errors='coerce')
                    .fillna(pd.Timestamp('1970-01-01')).astype(np.int64)).values
    order = np.argsort(ts_raw, kind='stable')
    ts_sort = ts_raw[order]
    if src_col and dst_col:
        mask = df[src_col].astype(str).values[order] != df[dst_col].astype(str).values[order]
        ts_clean = ts_sort[mask]
        n_loops = int((~mask).sum())
        if n_loops > 0: print(f'  Self-loops removidos: {n_loops}')
    else:
        ts_clean = ts_sort
    n_csv, n_cache = len(ts_clean), ei_all.shape[1]
    if n_csv != n_cache:
        raise AssertionError(f'Mismatch: {n_csv} ts vs {n_cache} arestas')
    ts_test = ts_clean[te_idx]
    print(f'  Timestamps: [{ts_test.min()}, {ts_test.max()}]')
    return ts_test

print('Funcoes base definidas.')

In [ ]:
# CELULA 3 - BTCS v3: WCC + Leiden Hierarquico + Budget
#
# Algoritmo hierarquico:
# 1. Top-k arestas → subgrafo de nos → WCC (cobertura alta como B0)
# 2. Para cada WCC com |induzidas| > B:
#    a. Construir Lk (temporal+estrutural) para as arestas top-k do componente
#    b. Leiden em Lk → sub-casos
# 3. Para cada WCC com |induzidas| <= B: manter como caso unico
# 4. Nos atribuidos por voto majoritario (conjuntos disjuntos entre casos)
# 5. Arestas induzidas vetorizado O(E)
# 6. Budget cap final → garante OCR(B) = 0

def build_Lk(src_sel, dst_sel, ts_sel, delta_L, max_hub_edges=500, verbose=True):
    """Constroi grafo auxiliar Lk. Retorna lista de arestas (i,j).
    Nos de Lk = arestas top-k. Aresta em Lk se compartilham no + |dt| <= delta_L."""
    K = len(src_sel)
    node_map = defaultdict(list)
    for i in range(K):
        node_map[int(src_sel[i])].append((i, int(ts_sel[i])))
        node_map[int(dst_sel[i])].append((i, int(ts_sel[i])))

    lk_edges = set()
    n_hubs_capped = 0
    for node, elist in node_map.items():
        n = len(elist)
        if n < 2:
            continue
        if n > max_hub_edges:
            n_hubs_capped += 1
            elist.sort(key=lambda x: x[1])
            elist = elist[:max_hub_edges]
        else:
            elist.sort(key=lambda x: x[1])
        for i in range(len(elist)):
            ei, ti = elist[i]
            for j in range(i+1, len(elist)):
                ej, tj = elist[j]
                if tj - ti > delta_L:
                    break
                if ei != ej:
                    lk_edges.add((min(ei, ej), max(ei, ej)))

    if verbose and n_hubs_capped > 0:
        print(f'    Lk: {n_hubs_capped} hubs capped at {max_hub_edges}')
    return list(lk_edges)


def build_btcs_cases(ranked, full, k=0.01, delta_L=7, resolution=1.0,
                     budget_B=100, seed=42):
    """BTCS v3: WCC → Leiden (hierarquico) → Budget.

    1. WCC no subgrafo de nos top-k  (cobertura = B0)
    2. Componente pequeno (|ind| <= B): manter como caso unico
    3. Componente grande  (|ind| >  B): Leiden em Lk local → sub-casos
    4. Nos atribuidos por voto majoritario (disjuntos)
    5. Arestas induzidas  (vetorizado O(E))
    6. Budget cap
    """
    scores = np.asarray(ranked['scores'], dtype=float)
    src    = np.asarray(ranked['src'],    dtype=np.int64)
    dst    = np.asarray(ranked['dst'],    dtype=np.int64)
    ts     = np.asarray(ranked['timestamps'], dtype=np.int64)
    sf     = np.asarray(full['src'],      dtype=np.int64)
    df_    = np.asarray(full['dst'],      dtype=np.int64)
    K      = max(1, int(math.ceil(k * len(scores))))
    sel    = np.argsort(-scores)[:K]

    src_sel = src[sel]
    dst_sel = dst[sel]
    ts_sel  = ts[sel]

    print(f'  K={K:,} top edges')

    # ─── Step 1: WCC no subgrafo de nos ───
    t0 = time.perf_counter()
    all_nodes = np.unique(np.concatenate([src_sel, dst_sel]))
    nmap = {int(n): i for i, n in enumerate(all_nodes)}
    n_compact = len(all_nodes)

    edges_compact = [(nmap[int(s)], nmap[int(d)])
                     for s, d in zip(src_sel, dst_sel)]
    g_node = ig.Graph(n=n_compact, edges=edges_compact, directed=False)
    g_node.simplify()
    wcc = g_node.connected_components(mode='weak')
    wcc_mem = np.array(wcc.membership, dtype=np.int64)
    n_wcc = int(wcc_mem.max()) + 1

    # Mapear cada aresta top-k → seu WCC (via no source)
    edge_wcc = np.array([wcc_mem[nmap[int(s)]] for s in src_sel],
                        dtype=np.int64)

    # Agrupar arestas top-k e nos por WCC
    wcc_edge_lists = [[] for _ in range(n_wcc)]
    wcc_node_sets  = [set() for _ in range(n_wcc)]
    for i in range(K):
        w = int(edge_wcc[i])
        wcc_edge_lists[w].append(i)
        wcc_node_sets[w].update([int(src_sel[i]), int(dst_sel[i])])

    t_wcc = time.perf_counter() - t0
    wcc_sizes = [len(e) for e in wcc_edge_lists if e]
    print(f'  WCC: {n_wcc:,} componentes ({t_wcc:.1f}s) | '
          f'median={np.median(wcc_sizes):.0f} max={max(wcc_sizes)}')

    # ─── Step 2: Contar induzidas por WCC ───
    t0 = time.perf_counter()
    max_node = int(max(sf.max(), df_.max(), all_nodes.max())) + 1
    wcc_gid = -np.ones(max_node, dtype=np.int64)
    for w, nodes in enumerate(wcc_node_sets):
        for nid in nodes:
            wcc_gid[nid] = w

    g_src_w = np.where(sf < max_node, wcc_gid[sf], -1)
    g_dst_w = np.where(df_ < max_node, wcc_gid[df_], -1)
    mask_w  = (g_src_w == g_dst_w) & (g_src_w >= 0)
    idx_w   = np.where(mask_w)[0]

    wcc_ind_count = np.zeros(n_wcc, dtype=np.int64)
    if len(idx_w) > 0:
        gw = g_src_w[idx_w]
        uq, cnt = np.unique(gw, return_counts=True)
        wcc_ind_count[uq] = cnt

    t_cnt = time.perf_counter() - t0
    n_small = int((wcc_ind_count[wcc_ind_count > 0] <= (budget_B or 1e18)).sum())
    n_large = int((wcc_ind_count > (budget_B or 1e18)).sum())
    print(f'  Induced por WCC ({t_cnt:.1f}s): {n_small} small (<= B), '
          f'{n_large} large (> B)')

    # ─── Step 3: Decidir split para WCCs grandes ───
    t0 = time.perf_counter()
    final_mem = np.full(K, -1, dtype=np.int64)
    next_id = 0
    n_kept = 0
    n_split = 0
    n_leiden_sub = 0

    for w in range(n_wcc):
        comp = wcc_edge_lists[w]
        if not comp:
            continue

        need_split = (budget_B is not None
                      and wcc_ind_count[w] > budget_B
                      and len(comp) >= 2)

        if not need_split:
            # Manter como caso unico
            for i in comp:
                final_mem[i] = next_id
            next_id += 1
            n_kept += 1
        else:
            # Refinar com Leiden no Lk local
            n_split += 1
            comp_arr = np.array(comp, dtype=np.int64)

            lk_local = build_Lk(
                src_sel[comp_arr], dst_sel[comp_arr], ts_sel[comp_arr],
                delta_L, verbose=False)

            if not lk_local:
                # Sem links temporais: cada aresta → caso proprio
                for i in comp:
                    final_mem[i] = next_id
                    next_id += 1
                n_leiden_sub += len(comp)
            else:
                n_local = len(comp)
                g_local = ig.Graph(n=n_local, edges=lk_local, directed=False)
                g_local.simplify()
                part = leidenalg.find_partition(
                    g_local, leidenalg.RBConfigurationVertexPartition,
                    resolution_parameter=resolution, seed=seed)
                local_mem = np.array(part.membership, dtype=np.int64)
                n_sub = int(local_mem.max()) + 1
                n_leiden_sub += n_sub

                for j, idx in enumerate(comp):
                    final_mem[idx] = next_id + int(local_mem[j])
                next_id += n_sub

    t_split = time.perf_counter() - t0
    n_total = next_id
    print(f'  Hierarchy ({t_split:.1f}s): {n_kept} kept + '
          f'{n_split} split ({n_leiden_sub} sub-cases) → {n_total} total')

    # ─── Step 4: Montar casos com nos disjuntos (voto majoritario) ───
    node_votes = defaultdict(lambda: defaultdict(int))
    for i in range(K):
        g = int(final_mem[i])
        if g < 0:
            continue
        node_votes[int(src_sel[i])][g] += 1
        node_votes[int(dst_sel[i])][g] += 1

    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []}
             for _ in range(n_total)]

    for nid, votes in node_votes.items():
        best = max(votes, key=votes.get)
        cases[best]['nodes'].add(nid)

    for i in range(K):
        g = int(final_mem[i])
        if g >= 0:
            cases[g]['seed_edges'].append(int(sel[i]))

    # Remover casos vazios
    cases = [c for c in cases if c['nodes']]

    # ─── Step 5: Arestas induzidas (vetorizado O(E)) ───
    t0 = time.perf_counter()
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g, case in enumerate(cases):
        for nid in case['nodes']:
            gid_of[nid] = g

    g_src_f = np.where(sf < max_node, gid_of[sf], -1)
    g_dst_f = np.where(df_ < max_node, gid_of[df_], -1)
    mask_f  = (g_src_f == g_dst_f) & (g_src_f >= 0)
    idx_f   = np.where(mask_f)[0]

    if len(idx_f) > 0:
        gf     = g_src_f[idx_f]
        order  = np.argsort(gf, kind='stable')
        g_sorted = gf[order]
        i_sorted = idx_f[order]
        uq, cnt  = np.unique(g_sorted, return_counts=True)
        for g_id, grp in zip(uq, np.split(i_sorted, np.cumsum(cnt)[:-1])):
            cases[g_id]['induced_edges'] = grp.tolist()

    t_ind = time.perf_counter() - t0

    # ─── Step 6: Budget cap ───
    n_capped = 0
    if budget_B is not None and budget_B > 0:
        for case in cases:
            if len(case['induced_edges']) > budget_B:
                n_capped += 1
                idx_arr = np.array(case['induced_edges'], dtype=np.int64)
                sc_ind = scores[idx_arr]
                top_b = idx_arr[np.argsort(-sc_ind)[:budget_B]]
                case['induced_edges'] = top_b.tolist()

    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    print(f'  Induced ({t_ind:.1f}s) | Budget B={budget_B}: '
          f'{n_capped}/{len(cases)} capped')
    print(f'  Final: {len(cases)} cases | '
          f'median={np.median(ind_sizes):.0f} '
          f'p95={np.percentile(ind_sizes, 95):.0f} '
          f'max={ind_sizes.max()}')

    return cases

print('BTCS v3 (WCC + Leiden hierarquico + Budget) definido.')

In [ ]:
# CELULA 4 - Carregar dados + timestamps
print('Carregando AML100k...')
ranked_100k, full_100k, gt_100k, te_idx_100k, ei_100k = load_dataset_artifacts(
    AML100K_ARTIF, AML100K_PROBS, AML100K_MODEL, AML100K_SEED, 'AML100k')
ts_100k = load_timestamps_from_csv(AML100K_BASE, te_idx_100k, ei_100k)
ranked_100k['timestamps'] = ts_100k
full_100k['timestamps']   = ts_100k

print('\nCarregando AML1M...')
ranked_1m, full_1m, gt_1m, te_idx_1m, ei_1m = load_dataset_artifacts(
    AML1M_ARTIF, AML1M_PROBS, AML1M_MODEL, AML1M_SEED, 'AML1M')
ts_1m = load_timestamps_from_csv(AML1M_BASE, te_idx_1m, ei_1m)
ranked_1m['timestamps'] = ts_1m
full_1m['timestamps']   = ts_1m

# Carregar resultados do nb01 para comparacao
df_baselines = pd.read_csv(NB01_OUT / 'b0_b1_b2_b3_results.csv')
print(f'\nnb01 baselines: {len(df_baselines)} linhas carregadas')
print('Dados prontos!')

In [ ]:
# CELULA 5 - Hiperparametros BTCS v3
DELTA_L     = 7       # janela temporal para Lk (dentro de WCCs grandes)
RESOLUTIONS = [0.5, 1.0, 2.0, 5.0]  # gamma do Leiden (so para WCCs > B)
BUDGET_B    = 100     # limite de arestas induzidas por caso
KS          = [0.01, 0.02, 0.05, 0.10]

print(f'delta_L={DELTA_L}  resolutions={RESOLUTIONS}')
print(f'budget_B={BUDGET_B}')
print(f'k: {[f"{int(k*100)}%" for k in KS]}')

In [ ]:
# CELULA 6 - Rodar BTCS v3: gamma x k x dataset
DATASETS = [
    ('AML100k', ranked_100k, full_100k, gt_100k),
    ('AML1M',   ranked_1m,   full_1m,   gt_1m),
]
rows = []

for ds, ranked, full, gt in DATASETS:
    print(f'\n{"="*50}  {ds}  {"="*50}')
    for gamma in RESOLUTIONS:
        for k in KS:
            kp = f'{int(k*100)}%'
            print(f'\n  g={gamma} k={kp}:')

            with measure_resources() as res:
                cases = build_btcs_cases(
                    ranked, full, k=k,
                    delta_L=DELTA_L, resolution=gamma,
                    budget_B=BUDGET_B, seed=42)
                m = evaluate_cases(cases, gt, ranked, k)

            method_name = f'BTCS_g{gamma}'
            row = {'dataset':ds, 'method':method_name,
                   'k%':kp, 'k_frac':k,
                   'delta_L':DELTA_L, 'resolution':gamma,
                   'budget_B':BUDGET_B,
                   **m, **res}
            rows.append(row)
            print(f"    {method_name}: #cases={m['n_cases']:,} "
                  f"cov={m['coverage']:.3f} pur={m['purity_induced']:.4f} "
                  f"OCR100={m['ocr_b100']:.3f} E/Ek={m['e_ind_over_ek']:.2f}x "
                  f"{res['time_s']:.1f}s")

df_btcs = pd.DataFrame(rows)
print(f'\nBTCS v3 concluido! {len(rows)} configuracoes.')

In [ ]:
# CELULA 7 - Tabela comparativa: B0 vs B1 vs BTCS_v3(B=100)
print('=== Tabela Comparativa: B0 vs B1 vs BTCS_v3(B=100) ===')

for ds in ['AML100k', 'AML1M']:
    print(f'\n{"="*40} {ds} {"="*40}')
    bl = df_baselines[df_baselines['dataset']==ds]

    comp_rows = []
    for kp in ['1%', '2%', '5%', '10%']:
        # B0
        b0r = bl[(bl['method']=='B0_WCC') & (bl['k%']==kp)]
        if not b0r.empty:
            r = b0r.iloc[0]
            comp_rows.append({'method':'B0_WCC','k%':kp,
                'n_cases':r['n_cases'],'coverage':r['coverage'],
                'purity':r['purity_induced'],'ocr_b100':r['ocr_b100'],
                'e_ind_over_ek':r['e_ind_over_ek'],'time_s':r['time_s']})
        # B1
        b1r = bl[(bl['method']=='B1_TemporalWCC') & (bl['k%']==kp)]
        if not b1r.empty:
            r = b1r.iloc[0]
            comp_rows.append({'method':'B1_Temporal','k%':kp,
                'n_cases':r['n_cases'],'coverage':r['coverage'],
                'purity':r['purity_induced'],'ocr_b100':r['ocr_b100'],
                'e_ind_over_ek':r['e_ind_over_ek'],'time_s':r['time_s']})
        # BTCS v3 (melhor gamma por coverage)
        sub = df_btcs[df_btcs['k%']==kp]
        sub = sub[sub['dataset']==ds].sort_values('coverage', ascending=False)
        if not sub.empty:
            r = sub.iloc[0]
            comp_rows.append({
                'method':f'BTCS_v3(g={r["resolution"]:.1f})',
                'k%':kp,
                'n_cases':r['n_cases'],'coverage':r['coverage'],
                'purity':r['purity_induced'],'ocr_b100':r['ocr_b100'],
                'e_ind_over_ek':r['e_ind_over_ek'],'time_s':r['time_s']})

    df_comp = pd.DataFrame(comp_rows)
    display(df_comp.round(4).pivot_table(
        index='k%', columns='method',
        values=['coverage','ocr_b100','e_ind_over_ek','purity'],
        aggfunc='first').round(4))

# Resumo: coverage gap v2 → v3
print('\n=== Melhoria Coverage: v2 (Leiden puro) → v3 (WCC+Leiden) ===')
try:
    df_v2 = pd.read_csv(BTCS_OUT / 'btcs_results_v2.csv')
    df_v2_b100 = df_v2[df_v2['budget_B']==100]
    for ds in ['AML100k', 'AML1M']:
        print(f'\n--- {ds} ---')
        for kp in ['1%', '2%', '5%', '10%']:
            v2 = df_v2_b100[(df_v2_b100['dataset']==ds) & (df_v2_b100['k%']==kp)]
            v3 = df_btcs[(df_btcs['dataset']==ds) & (df_btcs['k%']==kp)]
            if v2.empty or v3.empty:
                continue
            best_v2 = v2.sort_values('coverage', ascending=False).iloc[0]
            best_v3 = v3.sort_values('coverage', ascending=False).iloc[0]
            delta = best_v3['coverage'] - best_v2['coverage']
            print(f'  k={kp}: v2={best_v2["coverage"]:.3f} → '
                  f'v3={best_v3["coverage"]:.3f} ({"+" if delta>0 else ""}{delta:.3f})')
except FileNotFoundError:
    print('  (v2 CSV nao encontrado para comparacao)')

In [ ]:
# CELULA 8 - Plots: BTCS v3 vs baselines
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, ds in enumerate(['AML100k', 'AML1M']):
    bl = df_baselines[df_baselines['dataset']==ds]
    bt = df_btcs[df_btcs['dataset']==ds]

    # --- Plot 1: Coverage vs k (line plot) ---
    ax = axes[0, col]
    k_vals = [0.01, 0.02, 0.05, 0.10]

    for method, color, marker, label in [
        ('B0_WCC','#1f77b4','s','B0 WCC'),
        ('B1_TemporalWCC','#ff7f0e','^','B1 Temporal'),
        ('B2_WCC_Ctx','#2ca02c','v','B2 WCC+Ctx'),
        ('B3_HubPruned','#d62728','D','B3 Hub-Pruned')]:
        covs = []
        for kv in k_vals:
            kp = f'{int(kv*100)}%'
            row = bl[(bl['method']==method) & (bl['k%']==kp)]
            covs.append(float(row['coverage']) if not row.empty else np.nan)
        ax.plot(k_vals, covs, color=color, marker=marker, ms=8,
                label=label, alpha=0.7, ls='--')

    # BTCS v3: melhor gamma por k
    btcs_covs = []
    for kv in k_vals:
        kp = f'{int(kv*100)}%'
        sub = bt[bt['k%']==kp].sort_values('coverage', ascending=False)
        btcs_covs.append(float(sub.iloc[0]['coverage']) if not sub.empty else np.nan)
    ax.plot(k_vals, btcs_covs, color='black', marker='*', ms=14,
            label='BTCS v3 (B=100)', linewidth=2, zorder=10)

    ax.set_xlabel('k (fraction)')
    ax.set_ylabel('Coverage')
    ax.set_title(f'{ds} — Coverage vs k')
    ax.legend(fontsize=7, loc='lower left')
    ax.set_xticks(k_vals)
    ax.set_xticklabels(['1%','2%','5%','10%'])
    ax.grid(alpha=0.3)

    # --- Plot 2: OCR(100) barplot B0 vs B1 vs BTCS v3 ---
    ax = axes[1, col]
    labels_all = ['B0 WCC', 'B1 Temporal', 'BTCS v3']
    colors_all = ['#1f77b4', '#ff7f0e', '#2ca02c']
    x = np.arange(len(k_vals))
    w = 0.25

    for j, (method, color, label) in enumerate(zip(
            ['B0_WCC', 'B1_TemporalWCC', 'BTCS_best'], colors_all, labels_all)):
        vals = []
        for kv in k_vals:
            kp = f'{int(kv*100)}%'
            if method == 'BTCS_best':
                sub = bt[bt['k%']==kp].sort_values('coverage', ascending=False)
                vals.append(float(sub.iloc[0]['ocr_b100']) if not sub.empty else 0)
            else:
                row = bl[(bl['method']==method) & (bl['k%']==kp)]
                vals.append(float(row['ocr_b100']) if not row.empty else 0)
        ax.bar(x + j*w, vals, w, label=label, color=color, alpha=0.85)

    ax.set_title(f'{ds} — OCR(B=100)')
    ax.set_xticks(x + w)
    ax.set_xticklabels(['1%','2%','5%','10%'])
    ax.set_ylabel('fracao casos > 100 arestas')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(BTCS_OUT / 'btcs_v3_vs_baselines.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot salvo.')

In [ ]:
# CELULA 9 - Export
df_btcs.to_csv(BTCS_OUT / 'btcs_results_v3.csv', index=False)
print(f'CSV: {BTCS_OUT}/btcs_results_v3.csv')

# Tabela LaTeX: B0 vs B1 vs BTCS_v3 (melhor gamma)
cols = ['dataset','method','k%','n_cases','coverage','purity_induced',
        'cr_at_k','e_ind_over_ek','ocr_b100','time_s']
bl_sub = df_baselines[df_baselines['method'].isin(
    ['B0_WCC','B1_TemporalWCC'])][cols]

btcs_best_rows = []
for ds in ['AML100k','AML1M']:
    for kp in ['1%','2%','5%','10%']:
        sub = df_btcs[(df_btcs['dataset']==ds) & (df_btcs['k%']==kp)]
        sub = sub.sort_values('coverage', ascending=False)
        if not sub.empty:
            row = sub.iloc[0].copy()
            row['method'] = f"BTCS_v3(g={row['resolution']:.1f})"
            btcs_best_rows.append(row)
bt_sub = pd.DataFrame(btcs_best_rows)[cols]

df_final = pd.concat([bl_sub, bt_sub], ignore_index=True)
df_final[cols].round(4).to_latex(
    BTCS_OUT / 'btcs_comparison_table_v3.tex', index=False, escape=False)
print('LaTeX salvo.')

print('\nProgresso:')
print('1.[ok] nb00  2.[ok] nb01  3.[v3] nb02-BTCS')
print('4.[ ] nb03-ablacoes  5.[ ] nb04-multidataset')